<a href="https://colab.research.google.com/github/hoangson281205/ueh-ktlt-eco/blob/Gi%E1%BB%AFa-k%E1%BB%B3/Lab2(2_3_3)final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 2.3.3. Bài tập thực hành 1 — Xây dựng mô hình Naïve Bayes
Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu hành vi của khách hàng lấy tại
https://www.kaggle.com/code/arezalo/customer-behaviour-prediction-naive-bayes
Mục tiêu là dự đoán khả năng khách hàng mua sản phẩm dựa trên đặc điểm nhân khẩu học và hành vi.


#### Bước 1: Import thư viện cần thiết

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

print("import thành công")

import thành công


#### Bước 2: Tải file dữ liệu từ máy lên



In [ ]:
from google.colab import files

uploaded = files.upload()

# Đọc file vừa tải lên
df = pd.read_csv(list(uploaded.keys())[0])
print("oke")
display(df.head())

Saving Customer_Behaviour.csv to Customer_Behaviour.csv
oke


,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


#### Bước 3: Khám phá và kiểm tra thông tin dữ liệu

In [ ]:

print("Thông tin dữ liệu")
df.info()

print("Kiểm tra giá trị thiếu")
print(df.isnull().sum())

print("Thống kê mô tả dữ liệu")
display(df.describe(include='all'))

Thông tin dữ liệu
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   User ID          400 non-null    int64 
 1   Gender           400 non-null    object
 2   Age              400 non-null    int64 
 3   EstimatedSalary  400 non-null    int64 
 4   Purchased        400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB
Kiểm tra giá trị thiếu
User ID            0
Gender             0
Age                0
EstimatedSalary    0
Purchased          0
dtype: int64
Thống kê mô tả dữ liệu


,User ID,Gender,Age,EstimatedSalary,Purchased
count,4.000000e+02,400,400.000000,400.000000,400.000000
unique,NaN,2,NaN,NaN,NaN
top,NaN,Female,NaN,NaN,NaN
freq,NaN,204,NaN,NaN,NaN
mean,1.569154e+07,NaN,37.655000,69742.500000,0.357500
std,7.165832e+04,NaN,10.482877,34096.960282,0.479864
min,1.556669e+07,NaN,18.000000,15000.000000,0.000000
25%,1.562676e+07,NaN,29.750000,43000.000000,0.000000
50%,1.569434e+07,NaN,37.000000,70000.000000,0.000000
75%,1.575036e+07,NaN,46.000000,88000.000000,1.000000


**Nhận xét**

Tập dữ liệu Customer_Behaviour.csv gồm 400 quan sát và 5 biến, bao gồm:

User ID (định danh, không mang ý nghĩa dự báo),

Gender (biến phân loại),

Age, EstimatedSalary (biến định lượng),

Purchased (biến mục tiêu nhị phân).

Không phát hiện giá trị thiếu, các kiểu dữ liệu phù hợp. Dữ liệu nhìn chung sạch và đầy đủ, sẵn sàng cho giai đoạn tiền xử lý gồm loại bỏ biến định danh, mã hóa biến phân loại và chuẩn hóa dữ liệu số.


#### Bước 4: Xử lý dữ liệu trước khi xây dựng mô hình

In [ ]:
# Mã hóa biến phân loại 'Gender' nếu có
if 'Gender' in df.columns:
    le = LabelEncoder()
    df['Gender'] = le.fit_transform(df['Gender'])
    print("Đã mã hóa biến phân loại 'Gender'.")

# Xác định biến độc lập (features) và biến mục tiêu (target)
# Giả sử cột 'Purchased' là cột nhãn cần dự đoán
X = df.drop('Purchased', axis=1)
y = df['Purchased']

print("Dữ liệu sau xử lý")
display(df.head())


Đã mã hóa biến phân loại 'Gender'.
Dữ liệu sau xử lý


,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,1,19,19000,0
1,15810944,1,35,20000,0
2,15668575,0,26,43000,0
3,15603246,0,27,57000,0
4,15804002,1,19,76000,0


**Nhận xét**

Biến phân loại ‘Gender’ đã được mã hóa thành dạng số nhị phân (Male = 1, Female = 0), giúp mô hình Naïve Bayes có thể xử lý dữ liệu hiệu quả hơn.
Cấu trúc dữ liệu sau mã hóa giữ nguyên số quan sát và biến, đồng thời loại bỏ yếu tố ký tự, đảm bảo tính tương thích cho quá trình huấn luyện mô hình ở các bước tiếp theo.

#### Bước 5: Chia dữ liệu thành tập huấn luyện và kiểm tra

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Tập huấn luyện: {X_train.shape}")
print(f"Tập kiểm tra: {X_test.shape}")

Tập huấn luyện: (320, 4)
Tập kiểm tra: (80, 4)


**Nhận xét**

Dữ liệu được chia theo tỷ lệ 80% huấn luyện và 20% kiểm tra, tương ứng với 320 quan sát cho tập train và 80 quan sát cho tập test.
Tỷ lệ này đảm bảo đủ dữ liệu để mô hình học các đặc trưng tổng quát, đồng thời duy trì một phần dữ liệu độc lập để đánh giá khách quan hiệu quả dự đoán.

#### Bước 6: Xây dựng mô hình Naïve Bayes

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)
print(" Mô hình Naïve Bayes đã được huấn luyện thành công")

 Mô hình Naïve Bayes đã được huấn luyện thành công


#### Bước 7: Dự đoán và đánh giá mô hình

In [ ]:
y_pred = model.predict(X_test)

# Tính các chỉ số đánh giá
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Độ chính xác (Accuracy): {accuracy:.2f}")
print("Ma trận nhầm lẫn (Confusion Matrix):")
print(conf_matrix)
print("Báo cáo phân loại (Classification Report):")
print(class_report)

Độ chính xác (Accuracy): 0.86
Ma trận nhầm lẫn (Confusion Matrix):
[[47  4]
 [ 7 22]]
Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

           0       0.87      0.92      0.90        51
           1       0.85      0.76      0.80        29

    accuracy                           0.86        80
   macro avg       0.86      0.84      0.85        80
weighted avg       0.86      0.86      0.86        80



**Nhận xét**

Mô hình Naïve Bayes đạt độ chính xác tổng thể 86%, cho thấy khả năng dự đoán ở mức khá tốt trên tập kiểm tra.

Dựa trên báo cáo phân loại, lớp “không mua (0)” có precision = 0.87 và recall = 0.92, chứng tỏ mô hình nhận diện đúng phần lớn khách hàng không mua sản phẩm.
Ngược lại, lớp “mua (1)” có precision = 0.85 và recall = 0.76, phản ánh một số trường hợp khách hàng có mua bị dự đoán sai thành không mua.

Từ ma trận nhầm lẫn, có:

47 trường hợp “không mua” được dự đoán đúng,

4 trường hợp “không mua” bị nhầm là “mua”,

22 trường hợp “mua” được dự đoán đúng,

7 trường hợp “mua” bị nhầm là “không mua”.

Điều này cho thấy mô hình hoạt động tốt với nhóm “không mua”, nhưng còn bỏ sót một phần khách hàng tiềm năng, có thể do mất cân bằng dữ liệu hoặc đặc trưng hành vi giữa hai nhóm chưa được mô hình hóa đầy đủ.